In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt


In [3]:
# 1.2 Load Movielens dataset (example: 100k dataset)
# Replace 'ratings.csv' with the actual path to your dataset
ratings = pd.read_csv('MovieLens/ratings.csv')

# Preprocess data: Keep only 'userId', 'movieId', and 'rating' columns
ratings = ratings[['userId', 'movieId', 'rating']]
ratings.columns = ['user_id', 'item_id', 'rating']



In [18]:

# 1.3 Train-test split

if False:
    # Use part of the ratings dataset for training, for faster training
    # (optional, can be removed for full dataset training)
    ratings = ratings.sample(frac=0.5, random_state=42).reset_index(drop=True)

# Split data into train and test sets
# save some data for evaluation
# Split the data into train, test, and evaluation sets
traintest_data, eval_data = train_test_split(ratings, test_size=0.5) #, random_state=42)
train_data, test_data = train_test_split(traintest_data, test_size=0.2) #, random_state=42)

print(test_data.head(20))

       user_id  item_id  rating
10852       68     3114     3.0
13498       88      527     4.0
92200      597      368     4.0
53718      354     3176     4.0
28389      198      924     4.0
57957      380   102084     5.0
65747      424       47     5.0
21032      139    60684     4.0
12175       74    50274     4.0
98964      608     1799     4.5
55806      368     3578     3.0
54084      356    48516     4.0
38315      263     1784     4.0
41806      284      485     3.0
34040      232     3535     4.0
64778      414    71838     2.5
67715      438     1597     3.5
48101      312     1387     5.0
63694      414     4396     3.0
14731       93      168     4.0


In [19]:
# 1.4 Map user and item IDs to indices

# Extract unique users and items
n_users = ratings['user_id'].nunique()
n_items = ratings['item_id'].nunique()
print(f"Number of users: {n_users}, Number of items: {n_items}")
# Create user-item interaction matrix
user_id_map = {user_id: idx for idx, user_id in enumerate(ratings['user_id'].unique())}
item_id_map = {item_id: idx for idx, item_id in enumerate(ratings['item_id'].unique())}

# Map user and item IDs to indices
train_data['user_idx'] = train_data['user_id'].map(user_id_map)
train_data['item_idx'] = train_data['item_id'].map(item_id_map)
test_data['user_idx'] = test_data['user_id'].map(user_id_map)
test_data['item_idx'] = test_data['item_id'].map(item_id_map)
eval_data['user_idx'] = eval_data['user_id'].map(user_id_map)
eval_data['item_idx'] = eval_data['item_id'].map(item_id_map)

train_data.head(10)

Number of users: 610, Number of items: 9724


,user_id,item_id,rating,user_idx,item_idx
90356,587,2243,4.0,586,1580
40254,274,33672,3.5,273,3368
89954,584,344,1.0,583,470
88301,570,1214,3.5,569,75
65080,415,58559,4.0,414,238
86387,560,34162,3.5,559,1891
34455,232,40629,3.5,231,895
96771,603,2917,4.0,602,3031
58939,384,2959,4.0,383,192
8452,57,3175,4.0,56,415


In [21]:
# 1.5 Implementation of Matrix Factorization with SGD


class MatrixFactorization:
    def __init__(self, n_factors=50, learning_rate=0.01, reg_param=0.01, n_epochs=20, use_bias=True, verbose=True):
        """
        Initialize the Matrix Factorization model with optional bias terms.

        Parameters:
        - n_factors: Number of latent factors.
        - learning_rate: Learning rate for SGD.
        - reg_param: Regularization parameter.
        - n_epochs: Number of training epochs.
        - use_bias: Boolean flag to enable or disable bias terms.
        - verbose: Whether to print RMSE at each epoch.
        """
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.reg_param = reg_param
        self.n_epochs = n_epochs
        self.use_bias = use_bias
        self.verbose = verbose
        self.user_factors = None
        self.item_factors = None
        self.user_bias = None
        self.item_bias = None
        self.global_mean = None
        self.errors = []  # List to store RMSE at each epoch
        self.n_users = None
        self.n_items = None 
        self.user_id_map = None
        self.item_id_map = None

    def _initialize_factors(self, n_users, n_items, ratings):
        """
        Initialize user and item latent factors, biases, and global mean.
        """
        self.user_factors = np.random.normal(scale=1.0 / self.n_factors, size=(n_users, self.n_factors))
        self.item_factors = np.random.normal(scale=1.0 / self.n_factors, size=(n_items, self.n_factors))
        if self.use_bias:
            self.user_bias = np.zeros(n_users)
            self.item_bias = np.zeros(n_items)  
            self.global_mean = ratings['rating'].mean()

    def _predict(self, user_id, item_id):
        """
        Predict the rating for a given user and item.
        """
        prediction = np.dot(self.user_factors[user_id], self.item_factors[item_id])
        if self.use_bias:
            prediction += self.global_mean + self.user_bias[user_id] + self.item_bias[item_id]
        return prediction

    def _update_factors(self, user_id, item_id, rating):
        """
        Update user and item latent factors, biases, and global mean using SGD.
        """
        # Compute the prediction error
        prediction = self._predict(user_id, item_id)
        error = rating - prediction

        # Update user and item factors
        self.user_factors[user_id] += self.learning_rate * (error * self.item_factors[item_id] - self.reg_param * self.user_factors[user_id])
        self.item_factors[item_id] += self.learning_rate * (error * self.user_factors[user_id] - self.reg_param * self.item_factors[item_id])

        # Update user and item biases if bias is enabled
        if self.use_bias:
            self.user_bias[user_id] += self.learning_rate * (error - self.reg_param * self.user_bias[user_id])
            self.item_bias[item_id] += self.learning_rate * (error - self.reg_param * self.item_bias[item_id])


    def build_maps(self, data):

        # Extract unique users and items
        self.n_users = data['user_id'].nunique()
        self.n_items = data['item_id'].nunique()
        print(f"Number of users: {self.n_users}, Number of items: {self.n_items}")
        # Create mappings from user/item IDs to indices
        self.user_id_map = {user_id: idx for idx, user_id in enumerate(data['user_id'].unique())}
        self.item_id_map = {item_id: idx for idx, item_id in enumerate(data['item_id'].unique())}



    def fit(self, train_data, test_data, convergence_threshold=1e-4):
        """
        Train the Matrix Factorization model.

        Parameters:
        - train_data: DataFrame with columns ['user_id', 'item_id', 'rating'] for training.
        - test_data: DataFrame with columns ['user_id', 'item_id', 'rating'] for testing.
        """
        # joint train and test data to build maps
        if (self.n_users is None) or (self.n_items is None):
            print("ERROR: run build_maps(all_ratings) before fit()")
            #self.build_maps(pd.concat([train_data, test_data], ignore_index=True))
            return

        # Map user and item IDs to indices
        train_data['user_idx'] = train_data['user_id'].map(self.user_id_map)
        train_data['item_idx'] = train_data['item_id'].map(self.item_id_map)
        test_data['user_idx'] = test_data['user_id'].map(self.user_id_map)
        test_data['item_idx'] = test_data['item_id'].map(self.item_id_map)

        # Initialize latent factors
        self._initialize_factors(self.n_users, self.n_items, train_data)

        # Training loop
        for epoch in range(1, self.n_epochs + 1):
            # Shuffle training data for SGD
            train_data_shuffled = train_data.sample(frac=1).reset_index(drop=True)

            # Update latent factors using SGD
            for _, row in train_data_shuffled.iterrows():
                user_idx = int(row['user_idx'])  # Ensure user_idx is an integer
                item_idx = int(row['item_idx'])  # Ensure item_idx is an integer
                rating = row['rating']  # Rating can remain as float
                self._update_factors(user_idx, item_idx, rating)

            # Compute RMSE on test data
            test_predictions = [
                self._predict(int(user_idx), int(item_idx))
                for _, row in test_data.iterrows()
                for user_idx, item_idx in [(row['user_idx'], row['item_idx'])]
            ]
            test_ratings = test_data['rating'].values
            rmse = np.sqrt(mean_squared_error(test_ratings, test_predictions))
            # previous error
            prev_rmse = self.errors[-1] if self.errors else None
            # Store RMSE for this epoch
            self.errors.append(rmse)  # Store RMSE for this epoch
            # Check for convergence
            if prev_rmse is not None and abs(prev_rmse - rmse)/abs(rmse) < convergence_threshold:
                print(f"Converged at epoch {epoch}.")
                break
            # Check if error is increasing
            if prev_rmse is not None and rmse > prev_rmse:
                print(f"Error increased at epoch {epoch}. Stopping training.")
                break
            if self.verbose:
                print(f"Epoch {epoch}/{self.n_epochs} - Test RMSE: {rmse:.4f}")

    def get_errors(self):
        """
        Return the list of RMSE values at each epoch.
        """
        return self.errors

    def calculate_model_parameters(self):
        """
        Calculate the total number of parameters in the model.
        """
        n_users = self.user_factors.shape[0]
        n_items = self.item_factors.shape[0]

        # Parameters: user_factors, item_factors, user_bias, item_bias, global_mean
        total_params = (
            n_users * self.n_factors  # User latent factors
            + n_items * self.n_factors  # Item latent factors
            + (n_users if self.use_bias else 0)  # User biases
            + (n_items if self.use_bias else 0)  # Item biases
            + (1 if self.use_bias else 0)  # Global mean
        )
        return total_params

    def predict(self, user_id, item_id): #, user_id_map, item_id_map):
        """
        Predict the rating for a given user and item.

        Parameters:
        - user_id: User ID.
        - item_id: Item ID.
        - user_id_map: Mapping from user IDs to indices.
        - item_id_map: Mapping from item IDs to indices.
        """
        user_idx = self.user_id_map[user_id]
        item_idx = self.item_id_map[item_id]
        return self._predict(user_idx, item_idx)




In [24]:
# 1.6 Train the model

# Initialize and train the Matrix Factorization model
mf = MatrixFactorization(n_factors=5, learning_rate=0.01, reg_param=0.01, n_epochs=20, use_bias=False, verbose=True)
mf.build_maps(ratings)

# Train the model
mf.fit(train_data, test_data, convergence_threshold=1e-5)

# Calculate the number of parameters in the model
num_params = mf.calculate_model_parameters()
print(f"Total number of parameters in the model: {num_params}")


Number of users: 610, Number of items: 9724
Epoch 1/20 - Test RMSE: 3.6565
Epoch 2/20 - Test RMSE: 3.1391
Epoch 3/20 - Test RMSE: 2.2460
Epoch 4/20 - Test RMSE: 1.8600
Epoch 5/20 - Test RMSE: 1.6646
Epoch 6/20 - Test RMSE: 1.5495
Epoch 7/20 - Test RMSE: 1.4757
Epoch 8/20 - Test RMSE: 1.4255
Epoch 9/20 - Test RMSE: 1.3908
Epoch 10/20 - Test RMSE: 1.3650
Epoch 11/20 - Test RMSE: 1.3469
Epoch 12/20 - Test RMSE: 1.3326
Epoch 13/20 - Test RMSE: 1.3206
Epoch 14/20 - Test RMSE: 1.3140
Epoch 15/20 - Test RMSE: 1.3070
Epoch 16/20 - Test RMSE: 1.3037
Epoch 17/20 - Test RMSE: 1.2987
Epoch 18/20 - Test RMSE: 1.2973
Epoch 19/20 - Test RMSE: 1.2956
Epoch 20/20 - Test RMSE: 1.2928
Total number of parameters in the model: 51670


In [76]:
# Evaluate the model on the evaluation set
eval_data.describe()

,user_id,item_id,rating,user_idx,item_idx
count,50418.00000,50418.000000,50418.000000,50418.00000,50418.000000
mean,326.01908,19508.022056,3.499891,325.01908,1862.029275
std,182.80967,35709.329354,1.041303,182.80967,2007.240230
min,1.00000,1.000000,0.500000,0.00000,0.000000
25%,177.00000,1200.000000,3.000000,176.00000,462.000000
50%,325.00000,2997.000000,3.500000,324.00000,1137.000000
75%,477.00000,8264.750000,4.000000,476.00000,2507.000000
max,610.00000,193609.000000,5.000000,609.00000,9722.000000


In [77]:
# TEST
# mapping of items to indices
print(mf.item_id_map.keys().__len__())
keys = list(mf.item_id_map.keys())
print(keys[0:10])

for i in eval_data['item_id']:
    if i not in keys:
        print("Item not found in item_id_map: ", i)
        break

9724
[np.int64(1), np.int64(3), np.int64(6), np.int64(47), np.int64(50), np.int64(70), np.int64(101), np.int64(110), np.int64(151), np.int64(157)]


In [78]:
# 1.7 Prediction

# Example prediction
# Select one example user and item from the dataset
# Ensure that the user and item IDs exist in the dataset
rating_index = 200  # Change this to select a different rating
rating_row = ratings.iloc[rating_index]
user_id = rating_row['user_id'].astype(int)  # Ensure user_id is an integer
item_id = rating_row['item_id'].astype(int)  # Ensure item_id is an integer
print(f"User ID: {user_id}, Item ID: {item_id}")


predicted_rating = mf.predict(user_id, item_id) 
print(f"    Predicted rating for user {user_id} and item {item_id}: {predicted_rating:.2f}")

# Actual rating for comparison
actual_rating = rating_row['rating']
error = abs(predicted_rating - actual_rating)
print(f"    Actual rating for user {user_id} and item {item_id}: {actual_rating:.2f}, error: {error:.2f}")

User ID: 1, Item ID: 3052
    Predicted rating for user 1 and item 3052: 4.60
    Actual rating for user 1 and item 3052: 5.00, error: 0.40


In [25]:
# 1.8 Evaluate the model on the evaluation set

# Retrieve RMSE values
errors = mf.get_errors()
print("RMSE per epoch:", errors)


# Compute RMSE on test data
test_predictions = [
    mf.predict(user_id, item_id)
    for _, row in eval_data.iterrows()
    for user_id, item_id in [(row['user_id'], row['item_id'])]
]
test_ratings = eval_data['rating'].values
rmse = np.sqrt(mean_squared_error(test_ratings, test_predictions))

# Print RMSE
print("RMSE on evaluation data: ", rmse)





RMSE per epoch: [np.float64(3.6564556988533092), np.float64(3.139087318664244), np.float64(2.245979406110384), np.float64(1.8600299565455312), np.float64(1.6646031327630786), np.float64(1.5495415969359696), np.float64(1.4756532394313675), np.float64(1.4255025709565576), np.float64(1.3908094175452586), np.float64(1.3650207959347438), np.float64(1.3468946959512111), np.float64(1.3325700231702566), np.float64(1.3205895209822056), np.float64(1.313965330449248), np.float64(1.3069529720715767), np.float64(1.3037023348453785), np.float64(1.2987452266994053), np.float64(1.2973386349675196), np.float64(1.2955879529688263), np.float64(1.292751876102441)]
RMSE on evaluation data:  1.3192133071122583


In [80]:
# Predict values for all data in ratings, and plot the distribution of errors
def predict_errors_plot(input_data, model):
    """
    Predict ratings for the input data using the trained model.
    """
    predictions = input_data.copy()
    predictions['predicted_rating'] = predictions.apply(lambda row: model.predict(row['user_id'], row['item_id']), axis=1)

    # Calculate the error
    predictions['error'] = predictions['predicted_rating'] - predictions['rating']

    # Calculate RMSE for the entire dataset
    rmse_all = np.sqrt(mean_squared_error(predictions['rating'], predictions['predicted_rating']))
    print(f"RMSE error: {rmse_all:.4f}")

    # Plot the distribution of predicted ratings
    fig1 = plt.figure(figsize=(11, 3))
    
    plt.subplot(1, 2, 1)
    
    plt.hist(predictions['predicted_rating'], bins=50, alpha=0.5, color='green')

    # Plot the distribution of actual ratings
    plt.hist(predictions['rating'], bins=50, alpha=0.5, color='red')
    plt.title('Distribution of Actual and Predicted Ratings')
    plt.xlabel('Rating')
    plt.ylabel('Frequency')
    plt.grid()
    plt.legend(['Predicted Ratings', 'Actual Ratings'])
    plt.subplot(1, 2, 2)
    
    # Plot the distribution of errors
    #plt.figure(figsize=(8, 3))
    plt.hist(predictions['error'], bins=50, alpha=0.7, color='purple')
    plt.title('Distribution of Prediction Errors, {RMSE: %.4f}' % rmse_all)
    plt.axvline(rmse_all, color='black', linestyle='dashed', linewidth=1)
    plt.axvline(-rmse_all, color='black', linestyle='dashed', linewidth=1)
    plt.xlabel('Prediction Error')
    plt.ylabel('Frequency')
    plt.grid()
    plt.show()
    
    return predictions


In [81]:
# 1.9 Plot predicted ratings and errors
## pre = predict_errors_plot(eval_data, mf)

### 2. Testing models and parameters

In [ ]:
# 2.1 Plot RMSE vs. number of factors

factors_list = [1, 2, 5, 10, 30, 50]

errors_list = []
num_params_list = []

# Train and evaluate the model with different numbers of factors
for n_factors in factors_list:
    
    print(f"\n *******    Training with {n_factors} factors...")
    
    mf = MatrixFactorization(n_factors=n_factors, learning_rate=0.02, reg_param=0.01, n_epochs=20, use_bias=False, verbose=True)
    mf.build_maps(ratings)
    mf.fit(train_data, test_data)
    errors_list.append(mf.get_errors())

    # Evaluate the model on the evaluation set
    predict_errors_plot(eval_data, mf)
    
    # Calculate the number of parameters in the model
    num_params = mf.calculate_model_parameters()
    print(f"Total number of parameters in the model: {num_params}")    
    num_params_list.append(num_params)

# Plot RMSE vs. number of factors
plt.figure(figsize=(10, 6))
for i, n_factors in enumerate(factors_list):
    plt.plot(errors_list[i], label=f'Factors: {n_factors}')
plt.xlabel('Epochs')
plt.ylabel('RMSE')
plt.title('RMSE vs. Number of Factors')
plt.legend()
plt.grid()
plt.show()



 *******    Training with 1 factors...
Number of users: 610, Number of items: 9724
Epoch 1/20 - Test RMSE: 3.6194
Epoch 2/20 - Test RMSE: 2.6509
Epoch 3/20 - Test RMSE: 2.0710
Epoch 4/20 - Test RMSE: 1.8125
Epoch 5/20 - Test RMSE: 1.6697
Epoch 6/20 - Test RMSE: 1.5873
Epoch 7/20 - Test RMSE: 1.5325
Epoch 8/20 - Test RMSE: 1.4949
Epoch 9/20 - Test RMSE: 1.4709
Epoch 10/20 - Test RMSE: 1.4506
Epoch 11/20 - Test RMSE: 1.4333
Epoch 12/20 - Test RMSE: 1.4222
Epoch 13/20 - Test RMSE: 1.4125
Epoch 14/20 - Test RMSE: 1.4088
Epoch 15/20 - Test RMSE: 1.4023
Epoch 16/20 - Test RMSE: 1.3994
Epoch 17/20 - Test RMSE: 1.3924
Epoch 18/20 - Test RMSE: 1.3895
Converged at epoch 19.
Total number of parameters in the model: 10334

 *******    Training with 2 factors...
Number of users: 610, Number of items: 9724
Epoch 1/20 - Test RMSE: 3.5681
Epoch 2/20 - Test RMSE: 2.2138
Epoch 3/20 - Test RMSE: 1.7425
Epoch 4/20 - Test RMSE: 1.5609
Epoch 5/20 - Test RMSE: 1.4721
Epoch 6/20 - Test RMSE: 1.4209
Epoch 7/2

KeyboardInterrupt: 

In [ ]:
# 2.2. Plot RMSE vs. number of parameters, use Biased model




In [ ]:
# 2.3. Plot RMSE vs. learning rate
use_bias = True # False
learning_rates = [0.002, 0.005, 0.01, 0.02, 0.05]
n_factors = 30
n_epochs = 50





In [ ]:
# 2.4. Plot RMSE vs. regularization parameter
use_bias = True # False
reg_parameters = [0.001, 0.01, 0.1, 0.2]
n_factors = 30


